# EritroFase
## Corrección del fondo en imágenes de fase de eritrocitos mediante redes neuronales informadas por la física

**Curso:** SI3014 · Redes Neuronales y Aprendizaje Profundo · Universidad EAFIT
**Grupo 1:** Juan José Escobar · Emmanuel Hernández · Mateo Lopera
**Entrega:** 11 de noviembre de 2026
**Repositorio:** https://github.com/Jjes07/DeepLearning-Final

Este notebook es **autocontenido**: todo el código del proyecto (utilidades, modelo, pérdidas, métricas, líneas base y figuras) está aquí, explicado en markdown. Se entrega **ejecutado y con todas sus salidas**, para que cada afirmación de la presentación pueda verificarse sin volver a correr nada.

## 0. Guía de lectura

### 0.1 Cómo ejecutar
- Ejecutar las celdas **en orden**, de arriba abajo. Cada sección solo depende de las anteriores.
- Única dependencia externa: el dataset descomprimido en `data/raw/RBCs_Holograms/` (ver sección 2.1).
- Las semillas se fijan en la sección 2.2. Cada corrida guarda su configuración y sus métricas en `results/runs/<id>/`.
- Hardware, versiones y tiempos de ejecución se registran en las secciones 2.3 y 8.

### 0.2 Correspondencia con el contenido mínimo exigido
| Lineamiento del profesor | Sección del notebook | Componente de la rúbrica (peso) |
|---|---|---|
| A. Problema, justificación, objetivos | 1 | 1. Problema y objetivos (8) |
| B. Modelo, referencia y modificaciones | 4, 7 | 2. Arquitectura (13) |
| B. Base de datos y preprocesamiento | 3 | 3. Datos y preprocesamiento (13) |
| B. Hiperparámetros y equipo de cómputo | 2.3, 8 | 4. Entrenamiento (13) |
| C. Curvas de aprendizaje, sobreajuste/subajuste | 8 | 4. Entrenamiento (13) |
| C. Resultados cuantitativos e interpretación | 5, 6, 9 | 5. Resultados (13) |
| D. Datos no vistos y generalización | 10 | 6. Generalización (13) |
| E. Hallazgos, cumplimiento, limitaciones | 11, 12 | 7. Conclusiones (9) |

### 0.3 Criterios de rigor que sigue este trabajo
1. Cada experimento declara su **hipótesis antes** de ejecutarse, con un criterio de aceptación fijado de antemano, y la contrasta **después** con las salidas.
2. Se distingue lo que el diseño experimental **permite concluir** de lo que no: semillas, variantes que cambian varias cosas a la vez, diferencias dentro del ruido.
3. **Entrenamiento, validación y prueba están separados** y no comparten adquisiciones. Umbrales e hiperparámetros se eligen con validación; la prueba se usa una sola vez, al final.
4. Las comparaciones son **justas**: mismo conjunto, misma métrica, misma máscara de referencia y presupuesto declarado.
5. Toda cifra citada en el texto proviene de una salida visible del notebook.
6. Las figuras llevan ejes rotulados con unidades. Si una figura no aporta, se reemplaza por una tabla.
7. El lenguaje se ajusta a la evidencia: "es consistente con", no "demuestra".

### 0.4 Registro de hipótesis
Se actualiza al cerrar cada etapa.

| ID | Etapa | Hipótesis (antes de ejecutar) | Criterio de aceptación | Resultado | Sección |
|---|---|---|---|---|---|
| H1.1–H1.8 | E1 · Datos | Ver sección 3.0 | Ver sección 3.0 | Pendiente | 3 |

### 0.5 Registro de decisiones
| Decisión | Alternativas consideradas | Evidencia que la sustenta | Sección |
|---|---|---|---|
| Notebook autocontenido | Módulos en `src/` | Facilita la evaluación: un solo archivo con código, explicación y salidas | 0 |

## 1. Planteamiento del problema y objetivos  *(lineamiento A)*

### 1.1 Contexto: microscopía holográfica digital e imagen de fase cuantitativa
> ⏳ **Pendiente** — Qué es la DHM, qué codifica la fase y por qué un fondo no nulo dificulta el análisis de los eritrocitos.

### 1.2 Problema
> ⏳ **Pendiente** — Separar la contribución del fondo de la señal del eritrocito sin alterar esta última.

### 1.3 Justificación y pertinencia del deep learning
> ⏳ **Pendiente** — Por qué importa el problema y por qué una red informada por la física, frente a un ajuste clásico (Zernike o polinomio) que también se evalúa como línea base.

### 1.4 Objetivos (según la propuesta enviada el 31/10/2026)

**Objetivo general.** Desarrollar y entrenar una red neuronal informada por la física, basada en representaciones implícitas, para estimar y compensar el fondo en imágenes de fase de eritrocitos, reducir su fase residual hacia cero y obtener una visualización con fondo negro, preservando la morfología y el perfil de fase de las células.

**Objetivos específicos.**
1. Analizar los fundamentos matemáticos y físicos de las representaciones neuronales implícitas con activaciones sinusoidales y su relación con la codificación de Fourier, vinculándolos con la fase óptica y las aberraciones del sistema, para definir una arquitectura y restricciones de entrenamiento adecuadas para la estimación del fondo.
2. Caracterizar y seleccionar un conjunto representativo de imágenes de eritrocitos de la base de datos de fase, considerando su calidad, resolución, unidades, variación del fondo y disponibilidad de regiones sin células, para establecer los conjuntos de desarrollo y evaluación del modelo.
3. Validar cuantitativamente la reducción del fondo y la conservación del perfil de fase de los eritrocitos mediante métricas del fondo residual, errores de fase en la región celular y comparación de perfiles transversales, utilizando referencias disponibles o simulaciones con valores conocidos.

### 1.5 Alcance y ajustes respecto a la propuesta
> ⏳ **Pendiente** — Declarar con la evidencia de la sección 3 los ajustes al alcance: métricas en niveles de gris en lugar de radianes (el dataset no trae calibración) y uso de imágenes sintéticas solo si se justifica para la generalización.

## 2. Entorno de ejecución, reproducibilidad y utilidades generales

Esta sección prepara todo lo que usan las demás: librerías, rutas, semillas, dispositivo de cómputo y un reporte del equipo, que el lineamiento B exige presentar.

### 2.1 Importaciones y rutas del proyecto

Las rutas se construyen a partir de la carpeta donde está este notebook (`Path.cwd()`), así funcionan igual en Windows y en Mac. La celda falla con un mensaje claro si el dataset no está donde se espera.

In [ ]:
# 2.1 Importaciones y rutas del proyecto
import os
import sys
import re
import json
import time
import random
import platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import PIL
from PIL import Image
import torch

# Raíz del proyecto = carpeta que contiene este notebook
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw" / "RBCs_Holograms"
PHASE_DIR = DATA_DIR / "Phase"
SELECTION_DIR = ROOT / "data" / "selection"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
RUNS_DIR = ROOT / "results" / "runs"

for carpeta in (SELECTION_DIR, FIG_DIR, TAB_DIR, RUNS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

assert PHASE_DIR.exists(), (
    f"No se encontró {PHASE_DIR}. Descomprima el dataset en data/raw/RBCs_Holograms (ver README)."
)
print("Raíz del proyecto :", ROOT)
print("Carpeta de fases  :", PHASE_DIR)
print("Subcarpetas       :", sorted(p.name for p in PHASE_DIR.iterdir() if p.is_dir()))

### 2.2 Semillas y determinismo

`fijar_semillas` fija las semillas de Python, NumPy y PyTorch (CPU y GPU). Además se desactiva la búsqueda automática de algoritmos de cuDNN (`benchmark = False`) y se le pide usar los deterministas: el entrenamiento es algo más lento, pero dos ejecuciones con la misma semilla dan los mismos números. Cuando un experimento se repita con varias semillas, se llamará a `fijar_semillas(s)` antes de cada repetición.

In [ ]:
# 2.2 Semillas y determinismo
SEED = 42

def fijar_semillas(seed: int = SEED) -> None:
    """Fija las semillas de Python, NumPy y PyTorch para que los resultados sean repetibles."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

fijar_semillas(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Semilla global:", SEED)

### 2.3 Dispositivo de cómputo y reporte del equipo

`seleccionar_dispositivo` usa la GPU NVIDIA (CUDA) si existe, si no la GPU de Apple (MPS) y, en último caso, la CPU. `reporte_entorno` arma una tabla con sistema operativo, procesador, RAM, GPU y versiones de las librerías, y la guarda en `results/tables/entorno.csv`. Esa tabla es la que se cita en la presentación.

In [ ]:
# 2.3 Dispositivo de cómputo y reporte del equipo
def seleccionar_dispositivo() -> torch.device:
    """Devuelve CUDA si está disponible; si no, MPS (Mac); si no, CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = seleccionar_dispositivo()

def reporte_entorno() -> pd.DataFrame:
    """Tabla con el hardware y las versiones de software usadas en la ejecución."""
    filas = {
        "Sistema operativo": f"{platform.system()} {platform.release()}",
        "Procesador": platform.processor() or platform.machine(),
        "Núcleos lógicos de CPU": os.cpu_count(),
        "Python": sys.version.split()[0],
        "PyTorch": torch.__version__,
        "NumPy": np.__version__,
        "Pandas": pd.__version__,
        "Matplotlib": matplotlib.__version__,
        "Pillow": PIL.__version__,
        "Dispositivo usado": str(DEVICE),
    }
    try:
        import psutil
        filas["RAM total (GB)"] = round(psutil.virtual_memory().total / 1024**3, 1)
    except ImportError:
        filas["RAM total (GB)"] = "no disponible (instale psutil)"
    if DEVICE.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        filas["GPU"] = props.name
        filas["Memoria de la GPU (GB)"] = round(props.total_memory / 1024**3, 1)
        filas["CUDA (versión de PyTorch)"] = torch.version.cuda
    return pd.DataFrame.from_dict(filas, orient="index", columns=["Valor"])

entorno = reporte_entorno()
entorno.to_csv(TAB_DIR / "entorno.csv")
entorno

### 2.4 Utilidades generales: estilo de figuras y guardado de resultados
> ⏳ **Pendiente** — Funciones para guardar figuras y tablas con nombre de etapa, y un estilo común de matplotlib (ejes rotulados, unidades, tamaño de fuente). Se escriben al iniciar E1.

In [ ]:
# 2.4 Utilidades generales (se completa en E1)

## 3. Base de datos y preprocesamiento  *(lineamiento B · objetivo específico 2 · etapa E1)*

### 3.0 Hipótesis de la etapa (fijadas antes de ejecutar)

Definiciones que se usan abajo:
- **Recorte:** imagen PNG de 256×256 del dataset.
- **Variante:** la imagen base y sus versiones `-A` y `-B`.
- **Campo:** conjunto de recortes con la misma marca de tiempo y el mismo índice $X$ del nombre `Image__<fecha>__<hora>_<X>-<Y>[-A|-B]`.
- **Adquisición:** marca de tiempo `<fecha>__<hora>`; agrupa 4 campos.

| ID | Hipótesis | Criterio de aceptación (fijado de antemano) | Prueba | Sección |
|---|---|---|---|---|
| H1.1 | Todos los PNG de fase son de 8 bits, un canal y 256×256 píxeles | 100 % de las imágenes revisadas cumplen | Recorrido de los metadatos de los 36.864 archivos | 3.2 |
| H1.2 | `-A` y `-B` son volteos exactos (vertical y horizontal) de la imagen base; hay 12.288 recortes distintos | Diferencia píxel a píxel = 0 en el 100 % de una muestra aleatoria de 300 recortes | Comparación directa | 3.3 |
| H1.3 | Los 16 recortes $Y$ de un campo forman una cuadrícula contigua de 4×4 (1024×1024 píxeles) | Salto medio en las costuras ≤ 1,5 × el salto medio entre columnas vecinas del interior, en ≥ 95 % de 50 campos | Comparación de saltos | 3.3 |
| H1.4 | La partición original `Training`/`Validation` comparte recortes (fuga de información) | ≥ 1 recorte base presente en ambas carpetas | Cruce de nombres | 3.3 |
| H1.5 | La escala 0–255 se fijó por campo o imagen completa, no por recorte; por eso los niveles de gris son comparables dentro de un campo pero no necesariamente entre campos | H1.3 se cumple **y** los niveles medios del fondo difieren entre campos más que dentro de un campo | Estadísticas del fondo por campo | 3.4 |
| H1.6 | La fase no está envuelta: no hay saltos tipo $2\pi$ dentro de las células | < 0,1 % de pares de píxeles vecinos dentro de células con salto > 128 niveles, y perfiles continuos en las células de centro oscuro | Conteo de saltos y perfiles | 3.8 |
| H1.7 | El fondo no es constante: tiene una componente de baja frecuencia mayor que el ruido | $\sigma_{\text{fondo}} / \sigma_{\text{ruido}} > 2$ en la mayoría de las imágenes seleccionadas (definición en 3.9) | Ajuste polinomial del fondo | 3.9 |
| H1.8 | Hay variabilidad de calidad del fondo en el dataset, suficiente para separar un grupo de calidad media (desarrollo) y uno de menor calidad (generalización) | El índice de calidad del fondo muestra una cola superior identificable (definición en 3.5) | Distribución del índice sobre todos los recortes base | 3.5 |

Una hipótesis que no se cumpla **no se elimina**: se reporta como rechazada y se discuten sus consecuencias.

### 3.1 Origen y características del dataset
> ⏳ **Pendiente** — Resumen de Castañeda et al. (2024): equipo óptico, número de hologramas, recorte, aumento de datos y método de reconstrucción (Trujillo et al.).

### 3.2 Inventario del dataset completo: estructura, conteos, nomenclatura y formato (H1.1)
> ⏳ **Pendiente** — Funciones para leer la nomenclatura de los archivos y recorrer los metadatos, explicadas antes de usarlas.

In [ ]:
# 3.2 Inventario de archivos, análisis de la nomenclatura y formato de los PNG

### 3.3 Variantes aumentadas, campos de 1024×1024 y fuga en la partición original (H1.2, H1.3, H1.4)

In [ ]:
# 3.3 Verificación de volteos, continuidad entre recortes y cruce Training/Validation

### 3.4 Escala de grises y comparabilidad entre imágenes (H1.5)

In [ ]:
# 3.4 Niveles del fondo dentro de un campo y entre campos

### 3.5 Índice de calidad del fondo y criterios de selección de imágenes (H1.8)
> ⏳ **Pendiente** — Definición formal del índice (sin usar el modelo), distribución sobre todos los recortes base y criterios de selección: solo imágenes base (sin volteos), como máximo una por adquisición, estratificadas por calidad.

In [ ]:
# 3.5 Índice de calidad del fondo sobre los 12.288 recortes base

### 3.6 Selección y partición entrenamiento / validación / prueba / generalización
> ⏳ **Pendiente** — Tabla de conteos por conjunto y comprobación automática de que ninguna adquisición aparece en dos conjuntos. La lista se guarda en `data/selection/`.

In [ ]:
# 3.6 Selección de imágenes y partición por adquisición

### 3.7 Análisis exploratorio de las imágenes seleccionadas

Análisis de las imágenes que efectivamente usa el proyecto:
- dimensiones, tipo de dato y rango;
- histogramas de niveles de gris por conjunto;
- estadísticas por imagen: media, desviación, fracción de píxeles saturados en 0 y 255, fracción de fondo y número de células;
- ejemplos visuales de cada conjunto;
- comparación de las distribuciones entre entrenamiento, validación, prueba y generalización, para verificar que los conjuntos son comparables o documentar en qué difieren.

> ⏳ **Pendiente** — Se completa en E1.

In [ ]:
# 3.7.1 Dimensiones, tipo de dato y rango de las imágenes seleccionadas

In [ ]:
# 3.7.2 Histogramas de niveles de gris por conjunto

In [ ]:
# 3.7.3 Estadísticas por imagen y comparación entre conjuntos

In [ ]:
# 3.7.4 Ejemplos visuales por conjunto

### 3.8 ¿La fase está envuelta? (H1.6)

In [ ]:
# 3.8 Saltos entre píxeles vecinos dentro de las células y perfiles de las células de centro oscuro

### 3.9 Caracterización del fondo (H1.7)

In [ ]:
# 3.9 Ajuste polinomial del fondo, relación fondo/ruido y perfiles de ejemplo

### 3.10 Máscaras de referencia célula / fondo (independientes del modelo)
> ⏳ **Pendiente** — Método (umbral u otro), justificación y verificación visual. Estas máscaras solo se usan para evaluar, nunca para entrenar el modelo.

In [ ]:
# 3.10 Construcción y verificación visual de las máscaras de referencia

### 3.11 Preprocesamiento y su justificación

In [ ]:
# 3.11 Normalización y demás transformaciones aplicadas a la entrada

### 3.12 Contraste de las hipótesis de la etapa
> ⏳ **Pendiente** — Tabla hipótesis → evidencia → aceptada/rechazada, y qué se puede y qué no se puede concluir.

## 4. Fundamentos y formulación física  *(lineamiento B · objetivo específico 1 · etapa E2)*

### 4.1 Representaciones neuronales implícitas: SIREN y características de Fourier
> ⏳ **Pendiente** — Definiciones formales (SIREN, Sitzmann et al., 2020; Fourier features, Tancik et al., 2020), sesgo espectral y papel de $\omega_0$.

### 4.2 Modelo de formación de la imagen

Hipótesis de trabajo de la propuesta, que se comprueba con los datos:

$$
\varphi_{\text{obs}}(x,y) = \varphi_{\text{cel}}(x,y) + b(x,y) + \varepsilon(x,y),
\qquad
\varphi_{\text{corr}}(x,y) = \varphi_{\text{obs}}(x,y) - b_\theta(x,y)
$$

> ⏳ **Pendiente** — Definición de cada término y supuestos: suavidad del fondo y esparsidad de la muestra. Relación con la formulación de Huang y Cao (2023) y de Tang et al. (2023).

### 4.3 De las restricciones físicas a las funciones de pérdida
> ⏳ **Pendiente** — Cada término de la pérdida con su definición, su justificación física y su peso.

### 4.4 Código guía del curso
> ⏳ **Pendiente** — Copia del código guía del profesor (SIREN, mapa de confiabilidad, pérdidas, compensación de fondo), en celdas propias y con su procedencia indicada, para que el notebook no dependa de archivos externos.

In [ ]:
# 4.4.1 Código guía: siren.py

In [ ]:
# 4.4.2 Código guía: reliability.py y losses.py

In [ ]:
# 4.4.3 Código guía: unwrapper.py

### 4.5 Verificación del código guía con la fase sintética (error RMS)

In [ ]:
# 4.5 Ejecución del ejemplo sintético de main.py

### 4.6 Código guía sobre imágenes reales del dataset

In [ ]:
# 4.6 Aplicación del código guía a 1-2 recortes reales y diagnóstico

## 5. Líneas base no aprendidas
> ⏳ **Pendiente** — Ajuste de fondo de baja frecuencia (Zernike o polinomio, por mínimos cuadrados y con norma $L_1$ como en Huang y Cao) y `extract_sample_phase`, con hiperparámetros fijados en validación.

In [ ]:
# 5.1 Línea base: ajuste de fondo de baja frecuencia

In [ ]:
# 5.2 Línea base: extract_sample_phase del código guía

## 6. Protocolo de evaluación y métricas
> ⏳ **Pendiente** — Definiciones formales, con un ejemplo numérico de cada una:
- **Fondo residual:** media, desviación y RMS respecto a 0 en la región de fondo de referencia, antes y después.
- **Preservación de la célula:** diferencia y correlación de perfiles dentro de la célula.
- **Perfiles por cortes en línea recta** sobre los mismos píxeles antes y después.
- **Morfología:** Dice e IoU entre la máscara de referencia y el soporte de la imagen corregida.

In [ ]:
# 6.1 Implementación de las métricas y prueba con casos de valor conocido

## 7. Arquitectura del modelo  *(lineamiento B · etapa E3)*
### 7.1 Alternativas evaluadas
- **A:** red implícita optimizada por imagen (código guía; Tang et al., 2023).
- **B:** una sola red entrenada sobre el conjunto de entrenamiento.
- **C:** red que, a partir de la imagen, predice los parámetros del fondo.

> ⏳ **Pendiente** — Hipótesis, prueba rápida en validación y decisión, con sus limitaciones (semillas y presupuesto).

In [ ]:
# 7.1 Prueba rápida comparativa A / B / C

### 7.2 Arquitectura elegida: diagrama, bloques, dimensiones y parámetros entrenables
> ⏳ **Pendiente** — Diagrama, función de cada bloque, entrada y salida, conteo de parámetros, referencia original y modificaciones.

In [ ]:
# 7.2 Definición del modelo y resumen de parámetros

## 8. Entrenamiento y configuración experimental  *(lineamientos B y C · etapa E4)*

### 8.1 Hiperparámetros, función de pérdida, optimizador y su justificación

In [ ]:
# 8.1 Configuración de la corrida (se guarda en results/runs/<id>/config.json)

### 8.2 Entrenamiento

In [ ]:
# 8.2 Bucle de entrenamiento con registro de métricas de entrenamiento y validación

### 8.3 Curvas de aprendizaje (entrenamiento y validación) y análisis de sobreajuste o subajuste

In [ ]:
# 8.3 Curvas de pérdida y de métricas por época

### 8.4 Selección del mejor modelo, tiempos de entrenamiento e inferencia

In [ ]:
# 8.4 Criterio de selección en validación y tiempos medidos

## 9. Resultados cuantitativos en el conjunto de prueba  *(lineamiento C · objetivo específico 3 · etapa E5)*

### 9.1 Tabla de métricas: modelo frente a líneas base

In [ ]:
# 9.1 Métricas en prueba (media ± desviación por imagen)

### 9.2 Perfiles de intensidad por cortes antes y después

In [ ]:
# 9.2 Cortes en línea recta sobre los mismos píxeles, cruzando fondo y eritrocitos

### 9.3 Ejemplos visuales y mapas de error

In [ ]:
# 9.3 Entrada, fondo estimado, imagen corregida y visualización con fondo negro

### 9.4 Análisis de errores, fortalezas y limitaciones
> ⏳ **Pendiente** — Casos donde falla, sus posibles causas y lo que se puede y no se puede concluir.

## 10. Generalización con datos no vistos  *(lineamiento D)*
> ⏳ **Pendiente** — Procedencia de las muestras (adquisiciones no usadas y de menor calidad del fondo), en qué se diferencian, comparación con el desempeño en prueba y discusión de causas. Imágenes sintéticas solo si se justifica.

In [ ]:
# 10.1 Evaluación sobre el conjunto de generalización

In [ ]:
# 10.2 Comparación prueba frente a generalización y ejemplos

## 11. Discusión: compromiso entre desempeño, complejidad y tiempo
> ⏳ **Pendiente** — Tabla con métrica, parámetros, tiempo de entrenamiento e inferencia por método.

## 12. Conclusiones  *(lineamiento E)*
### 12.1 Principales hallazgos
> ⏳ **Pendiente** — 

### 12.2 Cumplimiento de los objetivos
| Objetivo | Evidencia (sección) | ¿Se cumplió? | Comentario |
|---|---|---|---|
| General | | | |
| Específico 1 | | | |
| Específico 2 | | | |
| Específico 3 | | | |

### 12.3 Limitaciones
> ⏳ **Pendiente** — 

### 12.4 Trabajo futuro
> ⏳ **Pendiente** — 

## 13. Referencias
1. R. Castañeda, C. Trujillo y A. Doblas (2024). A human erythrocytes hologram dataset for learning-based model training. *Data in Brief*, 54, 110424. https://doi.org/10.1016/j.dib.2024.110424
2. D. Xu, L. Cao y H. Sun (2025). Adaptive aberration-corrected quantitative phase microscopy via physics-informed deep learning. *Advanced Photonics*, 7(6), 066013. https://doi.org/10.1117/1.AP.7.6.066013
3. J. Tang, J. Zhang, S. Zhang, S. Mao, Z. Ren, J. Di y J. Zhao (2023). Phase aberration compensation via a self-supervised sparse constraint network in digital holographic microscopy. *Optics and Lasers in Engineering*, 168, 107671. https://doi.org/10.1016/j.optlaseng.2023.107671
4. Z. Huang y L. Cao (2023). Phase aberration separation for holographic microscopy by alternating direction sparse optimization. *Optics Express*, 31(8), 12520–12533. https://doi.org/10.1364/OE.488201
5. V. Sitzmann, J. N. P. Martel, A. W. Bergman, D. B. Lindell y G. Wetzstein (2020). Implicit Neural Representations with Periodic Activation Functions. arXiv:2006.09661. https://arxiv.org/abs/2006.09661
6. M. Tancik et al. (2020). Fourier Features Let Networks Learn High Frequency Functions in Low Dimensional Domains. arXiv:2006.10739. https://arxiv.org/abs/2006.10739
7. Código guía del curso (`main.py`, `siren.py`, `reliability.py`, `losses.py`, `unwrapper.py`, `test_siren.py`), incorporado en la sección 4.4.